In [1]:
from manim import *
import numpy as np

config.media_width = "100%"

In [2]:
%%manim -v WARNING -ql TCCDScene

np.random.seed(6)

x_min = -config.frame_width / 2
x_max = config.frame_width / 2
y_min = -config.frame_height / 2
y_max = config.frame_height / 2

BOUNDARY = (np.array([x_min, y_min]), np.array([x_max, y_max]))


def get_time_of_impact(p1, p2, t):
    dp = p2.position - p1.position
    dv = p2.velocity - p1.velocity
    r = p1.radius + p2.radius

    a = dv.dot(dv)
    b = 2 * dp.dot(dv)
    c = dp.dot(dp) - r * r

    discriminant = b * b - 4 * a * c

    if discriminant < 0:
        return None

    sqrt_disc = np.sqrt(discriminant)

    a2 = a * 2
    t0 = (-b - sqrt_disc) / a2
    t1 = (-b + sqrt_disc) / a2

    t_min = min(t0, t1)

    if 0 < t_min <= t:
        return t_min

    return None


def get_boundary_collision_time(
    p: Particle, boundary: tuple[tuple[float]], t_max: float
):
    pos = p.position
    vel = p.velocity
    r = p.radius
    t_hit = []

    x_min, y_min = boundary[0]
    x_max, y_max = boundary[1]

    if vel[0] < 0:
        t = (x_min + r - pos[0]) / vel[0]
        if 0 <= t <= t_max:
            t_hit.append(t)
    elif vel[0] > 0:
        t = (x_max - r - pos[0]) / vel[0]
        if 0 <= t <= t_max:
            t_hit.append(t)

    if vel[1] < 0:
        t = (y_min + r - pos[1]) / vel[1]
        if 0 <= t <= t_max:
            t_hit.append(t)
    elif vel[1] > 0:
        t = (y_max - r - pos[1]) / vel[1]
        if 0 <= t <= t_max:
            t_hit.append(t)

    return min(t_hit) if t_hit else None


def resolve_boundary_collision(
    p: Particle, t_impact: float, grid: tuple[tuple[float]]
):
    p.position += p.velocity * t_impact

    x_min, y_min = grid[0]
    x_max, y_max = grid[1]
    r = p.radius

    if p.position[0] - r <= x_min and p.velocity[0] < 0:
        p.velocity[0] *= -1
    elif p.position[0] + r >= x_max and p.velocity[0] > 0:
        p.velocity[0] *= -1

    if p.position[1] - r <= y_min and p.velocity[1] < 0:
        p.velocity[1] *= -1
    elif p.position[1] + r >= y_max and p.velocity[1] > 0:
        p.velocity[1] *= -1


def resolve_particle_collision(p1, p2, t_min):
    p1.position += p1.velocity * t_min
    p2.position += p2.velocity * t_min

    n = p2.position - p1.position
    dist2 = n.dot(n)
    if dist2 == 0:
        return

    n_hat = n / np.sqrt(dist2)
    v_rel = p2.velocity - p1.velocity
    v_rel_n = v_rel.dot(n_hat)

    if v_rel_n >= 0:
        return

    m1, m2 = p1.mass, p2.mass
    impulse = (2 * m1 * m2 / (m1 + m2)) * v_rel_n * n_hat

    p1.velocity += impulse / m1
    p2.velocity -= impulse / m2


class Particle:
    def __init__(self, position, velocity, radius=0.25, mass=1.0):
        self.position = np.array(position, dtype=float)
        self.velocity = np.array(velocity, dtype=float)
        self.radius = radius
        self.mass = mass

    def manim_circle(self, color=BLUE, alpha=1):
        return Circle(
            radius=self.radius, color=color, fill_opacity=alpha
        ).move_to(np.array([self.position[0], self.position[1], 0]))

    def manim_arrow(self, t: float, color=RED, alpha=1):
        start = np.append(self.position[:2], 0)
        end = np.append(self.position[:2] + self.velocity[:2] * t, 0)
        return Arrow(
            start,
            end,
            buff=0,
            color=color,
            fill_opacity=alpha,
            stroke_width=1,
            tip_length=0.2,
        )


class TCCDScene(Scene):
    def construct(self):
        particles: list[Particle] = []

        def generate_random_points(x_min, y_min, x_max, y_max):
            x_low, x_high = sorted([x_min, x_max])
            y_low, y_high = sorted([y_min, y_max])

            x = np.random.uniform(x_low + 1, x_high - 1)
            y = np.random.uniform(y_low + 1, y_high - 1)
            return np.array([x, y])

        def generate_high_velocity(min_speed=2.0, max_speed=5.0):
            angle = np.random.uniform(0, 2 * np.pi)
            speed = np.random.uniform(min_speed, max_speed)
            return np.array([speed * np.cos(angle), speed * np.sin(angle)])

        # Initialize the particles
        for _ in range(10):
            pos = generate_random_points(*BOUNDARY[0], *BOUNDARY[1])
            vel = generate_high_velocity()

            particle = Particle(pos, vel)
            particles.append(particle)

        # Display the particles
        self.play(*[GrowFromCenter(p.manim_circle()) for p in particles])
        self.play(*[GrowArrow(p.manim_arrow(1)) for p in particles])

        self.wait()

        self.tccd(BOUNDARY, particles, 1)

        self.wait(2)

    def tccd(
        self,
        boundary: tuple[tuple[float]],
        particles: list[Particle],
        t: float,
    ):
        count = 0

        circles = {p: p.manim_circle(alpha=0.25) for p in particles}

        trajectories = [p.manim_circle(alpha=0.25) for p in particles]

        while True:
            collisions: list[(float, (Particle, Particle | str))] = []

            count += 1

            # 1. Detect all valid particle-particle collisions
            for i in range(len(particles)):
                for j in range(i + 1, len(particles)):
                    p1, p2 = particles[i], particles[j]

                    local_t = get_time_of_impact(p1, p2, t)
                    if local_t is not None:
                        collisions.append((local_t, (p1, p2)))

            # 2. Detect boundary collisions
            for p in particles:
                bt = get_boundary_collision_time(p, boundary, t)
                if bt is not None:
                    collisions.append((bt, (p, "boundary")))

            if not collisions:
                break  # Stop the loop since there are no more collisions

            # 3. Select and resolve earliest collision
            t_min, pair = min(collisions, key=lambda x: x[0])

            if isinstance(pair[1], str) and pair[1] == "boundary":
                p = circles[pair[0]]
                resolve_boundary_collision(pair[0], t_min, boundary)
                trajectories.append(
                    Line(
                        p.get_center(),
                        [*pair[0].position, 0],
                        color=RED,
                        stroke_width=1,
                    )
                )
                self.play(p.animate.set_color(GREEN))
                self.play(p.animate.move_to([*pair[0].position, 0]))
                self.play(GrowArrow(pair[0].manim_arrow(t - t_min)))
                self.play(p.animate.set_color(BLUE))
            else:
                p1 = circles[pair[0]]
                p2 = circles[pair[1]]
                resolve_particle_collision(pair[0], pair[1], t_min)
                trajectories.append(
                    Line(
                        p1.get_center(),
                        [*pair[0].position, 0],
                        color=RED,
                        stroke_width=1,
                    )
                )
                trajectories.append(
                    Line(
                        p2.get_center(),
                        [*pair[1].position, 0],
                        color=RED,
                        stroke_width=1,
                    )
                )
                self.play(p1.animate.set_color(GREEN), p2.animate.set_color(GREEN))
                self.play(
                    p1.animate.move_to([*pair[0].position, 0]),
                    p2.animate.move_to([*pair[1].position, 0]),
                )
                self.play(
                    GrowArrow(pair[0].manim_arrow(t - t_min)),
                    GrowArrow(pair[1].manim_arrow(t - t_min)),
                )
                self.play(
                    p1.animate.set_color(BLUE), p2.animate.set_color(BLUE)
                )

            t -= t_min

            for p in particles:
                if p in pair:
                    continue

                p.position = p.position + p.velocity * t_min
                trajectories.append(
                    Line(
                        circles[p].get_center(),
                        [*p.position, 0],
                        color=RED,
                        stroke_width=1,
                    )
                )

            self.play(
                *[
                    circles[p].animate.move_to([*p.position, 0])
                    for p in particles
                    if p not in pair
                ]
            )

        # 4. Move all particles to final position
        final_positions = [p.manim_circle(color=RED) for p in particles]

        self.play(*[FadeIn(c) for c in final_positions])

        for p in particles:
            p.position = p.position + p.velocity * t
            trajectories.append(
                Line(
                    circles[p].get_center(),
                    [*p.position, 0],
                    color=RED,
                    stroke_width=1,
                )
            )

        self.play(
            *[
                c.animate.move_to([*p.position, 0])
                for (c, p) in zip(final_positions, particles)
            ],
            rate_func=linear,
        )

        self.add(*trajectories)
        self.fade_out_except([*final_positions, *trajectories])

        self.play(*[c.animate.set_color(BLUE) for c in final_positions])

    def fade_out_except(self, keep_list):
        all_objs = Group(*self.mobjects)
        to_fade = [m for m in all_objs if m not in keep_list]
        self.play(*[FadeOut(m) for m in to_fade])

Manim Community v0.19.0